# Naive Algorithm for Solving Simple Sudoku #

$Author$: Michael Simons

$Class$: MTH 337 - Mathematical & Scientific Computing

$Date$: 12/4/22

## Introduction ##

Sudoku is a logic-based puzzle where the goal is to fill a 9x9 grid with digits 1 to 9, such that each row, column, and 3x3 subgrid contains all the digits. Simple Sudoku puzzles are a subset of Sudoku puzzles that can be solved using a straightforward backtracking algorithm.

The algorithm takes a partially filled Sudoku puzzle as input and attempts to fill in the empty cells with valid numbers, using set intesections to represent the constraints of rows, columns, and 3x3 boxes. The goal is to demonstrate a solution to simple Sudoku puzzles using a straightforward backtracking approach.

In [1]:
import requests                #Will be used to request text representing the boards
import re                      #Will be use to parse the requested string
import numpy as np             #Will be used to represent sudoku boards

board_data=requests.get("https://scassani.github.io/MTH337-F22/_downloads/901e92afbe26b26389235823a5a17ec2/sudokus.txt").text
temp_boards=[]                                        #Initialize structure to store the unformatted boards
for i in board_data.split('\r\n\r\n'):                #Split each sudoku board into separate strings
    temp_boards.append(i.split('\r\n'))               #Split each board's rows into separate strings
    
sudoku_boards=[]                                      #Initialize structure to fill with formatted boards
for board in temp_boards:                             #Iterate through each board
    new_board=[]                                      #Initialize new board to fill
    if '' in board:board.remove('')                   #Check for and remove empty lines in each board
    for row in board:                                 #Iterate through each row,
        new_row=[]                                    #Create a new row to fill
        for i in row.replace(' ',''):                 #Remove spaces
            new_row.append(int(i))                    #Fill new row with integer values
        new_board.append(new_row)                     #Add new row to board
    sudoku_boards.append(new_board)                   #Add each board to the list of boards
    
    
#Given a sudoku puzzle s and the row index i, returns a set of the numbers which could fit in that row
def check_row(s,i):
    one_to_nine=set(range(1,10))                      #Initialize a set of the integers 1-9
    for k in set(s[i]):                               #Iterate through each number already found in the row
        if k in one_to_nine:                          #Remove found numbers from the initialized set
            one_to_nine.remove(k)
    return one_to_nine                                #Return the remaining possibilities

#Given a sudoku puzzle s and the column index j, returns a set of the numbers which could fit in that row
def check_col(s,j):
    one_to_nine=set(range(1,10))                      #Initilize a set of the integers 1-9
    for k in set([row[j] for row in s]):              #Iterate through each number already found in the column
        if k in one_to_nine:                          #Remove found numbers from the initialized set
            one_to_nine.remove(k)
    return one_to_nine                                #Return the remaining possibilities

#Given a sudoku puzzle s and the row and column indices i,j, returns a set of the numbers which could fit in the 3x3 box
def check_box(s,i,j):
    box_row_init=i-(i%3)                              #Calculate the box's starting row index by subtracting the remainder
    box_col_init=j-(j%3)                              #Calculate the box's starting column index by subtracting the remainder
    one_to_nine=set(range(1,10))                      #Initialize a set of the integers 1-9
    for r in range(box_row_init,box_row_init+3):      #Iterate through rows of the box
        for c in range(box_col_init,box_col_init+3):  #Iterate through columns of the box
            if s[r][c] in one_to_nine:                #Remove found numbers from the initialized set
                one_to_nine.remove(s[r][c])
    return one_to_nine                                #Return the remaining possibilities

#Boolean to determine whether or not a board has been solved
def solved(s):
    for row in range(0,len(s)):
        for col in range(0,len(s)):
            if s[row][col]==0:           #Return false if a 0 is present on the board
                return False
    return True                          #Return true if no 0 present

#Solves a valid simple sudoku game and displays the solution
def solve_sudoku(s):
    while not solved(s):
        for row in range(0,9):
            for col in range(0,9):
                if s[row][col]==0:
                    possibilities=check_row(s,row).intersection(check_col(s,col).intersection(check_box(s,row,col)))
                    if(len(possibilities)==1):
                        s[row][col]=list(possibilities)[0]
                        return solve_sudoku(s)
    return print_sudoku(np.array(s))

#Displays a sudoku board given as a 9x9 numpy array
def print_sudoku(arr):
    for i in range(9):
        for j in range(9):
            x = arr[i, j] if arr[i, j] != 0 else "."
            print(f" {x} ", end="")
            if j in [2, 5]:
                print("\u2551", end="")
        print("")
        if i in [2, 5]:
            print("\u2550"*9 + "\u256C" + "\u2550"*9 + "\u256C" + "\u2550"*9)

The solve_sudoku function implements the backtracking algorithm to solve the Sudoku puzzle. It iterates through each cell of the board and checks if the cell is empty (represented by 0). If a cell is empty, it calculates the possible numbers that can be placed in that cell based on the constraints of rows, columns, and 3x3 boxes. 

If only one possibility exists, the algorithm assigns that number to the cell and recursively calls itself to continue solving the puzzle. If multiple possibilities exist, the algorithm backtracks and tries different possibilities for previous cells. The solved function checks if the board is complete by verifying if all cells have a non-zero value.

The following code demonstrates the functionality of the solve_sudoku function by solving a set of Sudoku puzzles retrieved from an online text file. The code first retrieves the puzzles, parses them into 2D arrays, and then calls the solve_sudoku function on each puzzle. The results of the solved puzzles are then printed.

In [2]:
for board_num in range(1,len(sudoku_boards)+1):                     #Iterate through previously stored boards
    print(f'\nSudoku#{board_num} (unsolved): ')                     #Display unsolved board
    print_sudoku(np.array(sudoku_boards[board_num-1]))
    print(f'\nSudoku#{board_num} (solved): ')                       #Display solved board
    solve_sudoku(sudoku_boards[board_num-1])
    print('\n')


Sudoku#1 (unsolved): 
 .  4  . ║ .  .  . ║ 1  7  9 
 .  .  2 ║ .  .  8 ║ .  5  4 
 .  .  6 ║ .  .  5 ║ .  .  8 
═════════╬═════════╬═════════
 .  8  . ║ .  7  . ║ 9  1  . 
 .  5  . ║ .  9  . ║ .  3  . 
 .  1  9 ║ .  6  . ║ .  4  . 
═════════╬═════════╬═════════
 3  .  . ║ 4  .  . ║ 7  .  . 
 5  7  . ║ 1  .  . ║ 2  .  . 
 9  2  8 ║ .  .  . ║ .  6  . 

Sudoku#1 (solved): 
 8  4  5 ║ 6  3  2 ║ 1  7  9 
 7  3  2 ║ 9  1  8 ║ 6  5  4 
 1  9  6 ║ 7  4  5 ║ 3  2  8 
═════════╬═════════╬═════════
 6  8  3 ║ 5  7  4 ║ 9  1  2 
 4  5  7 ║ 2  9  1 ║ 8  3  6 
 2  1  9 ║ 8  6  3 ║ 5  4  7 
═════════╬═════════╬═════════
 3  6  1 ║ 4  2  9 ║ 7  8  5 
 5  7  4 ║ 1  8  6 ║ 2  9  3 
 9  2  8 ║ 3  5  7 ║ 4  6  1 



Sudoku#2 (unsolved): 
 8  .  2 ║ .  5  . ║ 7  .  1 
 .  .  7 ║ .  8  2 ║ 4  6  . 
 .  1  . ║ 9  .  . ║ .  .  . 
═════════╬═════════╬═════════
 6  .  . ║ .  .  1 ║ 8  3  2 
 5  .  . ║ .  .  . ║ .  .  9 
 1  8  4 ║ 3  .  . ║ .  .  6 
═════════╬═════════╬═════════
 .  .  . ║ .  .  4 ║ .  2  . 
 

## Conclusion ##

This naive backtracking algorithm effectively solves simple Sudoku puzzles by systematically exploring possible values for each empty cell. While it demonstrates a straightforward approach to solving Sudoku, its time complexity is exponential, making it impractical for solving complex Sudoku puzzles with many empty cells. The algorithm's naiveness lies in its exhaustive search strategy, which can lead to significant computational effort, especially for larger puzzles. This approach serves as a fundamental introduction to Sudoku solving techniques, highlighting the importance of constraint satisfaction and backtracking, but more efficient algorithms are required for tackling more challenging Sudoku puzzles.